In [1]:
import pandas as pd
import numpy as np
from grape import EmbeddingResult
import pickle

/home/sanya/.conda/envs/kg-embed/lib/python3.8/site-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (1.26.9) or chardet (5.1.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn(
/home/sanya/.conda/envs/kg-embed/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load embeddings from DGLKE results as GRAPE embeddings and save

### Load mapping files

In [2]:
entity_map_file = '../data/npkg/entities.tsv'
relation_map_file = '../data/npkg/relations.tsv'

In [3]:
entities_df = pd.read_csv(entity_map_file, sep=',', header=None, names=['id', 'entity'])
entities_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1089125 entries, 0 to 1089124
Data columns (total 2 columns):
 #   Column  Non-Null Count    Dtype 
---  ------  --------------    ----- 
 0   id      1089125 non-null  int64 
 1   entity  1089125 non-null  object
dtypes: int64(1), object(1)
memory usage: 16.6+ MB


In [4]:
relations_df = pd.read_csv(relation_map_file, sep=',', header=None, names=['id', 'relation'])
relations_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1343 entries, 0 to 1342
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        1343 non-null   int64 
 1   relation  1343 non-null   object
dtypes: int64(1), object(1)
memory usage: 21.1+ KB


In [5]:
node_mapping = {row['id']: row['entity'] for _, row in entities_df.iterrows()}
edge_type_mapping = {row['id']: row['relation'] for _, row in relations_df.iterrows()}

### Load embeddings

In [6]:
EMBED_DIR = '../dglke-modeling/output/'

In [12]:
def load_embeddings(model_name):
    print('Loading embeddings for model:', model_name)
    EMBED_PATH = EMBED_DIR + model_name + '_npkg_embeddings_all_data/'
    EMBED_ENTITY_FILENAME = 'npkg_'+model_name+'_entity.npy'
    EMBED_RELATION_FILENAME = 'npkg_'+model_name+'_relation.npy'
    entity_embedding = np.load(EMBED_PATH + EMBED_ENTITY_FILENAME)
    relation_embedding = np.load(EMBED_PATH + EMBED_RELATION_FILENAME)
    print('Embeddings loaded')
    return entity_embedding, relation_embedding

In [13]:
def create_df_from_embeddings(entity_embedding, relation_embedding):
    print('Processing embeddings to dataframe')
    node_embeddings_df = pd.DataFrame(entity_embedding, index=entities_df['entity'])
    edge_type_embeddings_df = pd.DataFrame(relation_embedding, index=relations_df['relation'])
    return node_embeddings_df, edge_type_embeddings_df

In [14]:
def get_embedding_result(model_name):
    entity_embedding, relation_embedding = load_embeddings(model_name)
    node_embeddings_df, edge_type_embeddings_df = create_df_from_embeddings(entity_embedding, relation_embedding)
    embedding_result = EmbeddingResult(
        embedding_method_name=model_name,
        node_embeddings=node_embeddings_df,
        edge_type_embeddings=edge_type_embeddings_df
    )
    print('Embedding result created')
    return embedding_result

In [17]:
##change model name before running
model_name = 'ComplEx'
embedding_result = get_embedding_result(model_name)
embedding_result_dict = embedding_result.dump()
with open('embeddings/npkg_'+model_name+'_embeddings.pickle', 'wb') as f:
    pickle.dump(embedding_result_dict, f)

Loading embeddings for model: ComplEx
Embeddings loaded
Processing embeddings to dataframe
Embedding result created


In [11]:
##load and check embedding result
with open('embeddings/npkg_'+model_name+'_embeddings.pickle', 'rb') as f:
    embedding_result_dict = pickle.load(f)
embedding_result = EmbeddingResult.load(embedding_result_dict)
embedding_result.get_node_embedding_from_index(0)

,0,1,2,3,4,5,6,7,8,9,...,390,391,392,393,394,395,396,397,398,399
entity,,,,,,,,,,,,,,,,,,,,,
clo:0035452,-0.017144,0.026280,0.016505,-0.026712,0.011255,-0.016669,0.000365,-0.004368,-0.017974,-0.027560,...,0.024926,0.018848,-0.003885,0.008644,0.009808,0.008074,0.006638,-0.024671,0.012864,-0.009459
clo:0000374,-0.004602,-0.027027,-0.004305,-0.013631,-0.019567,0.011372,0.041273,-0.027809,-0.009674,-0.007480,...,-0.006057,0.018146,-0.021681,0.014010,-0.033831,-0.021717,0.037921,-0.008668,0.017929,0.005895
ensembl:ENST00000256362,-0.022936,-0.014642,-0.033662,-0.035045,0.016510,-0.004701,-0.011429,0.026532,0.026691,-0.001963,...,0.032194,-0.011674,-0.036820,-0.016718,0.007127,-0.001612,-0.004708,0.015629,0.035567,-0.012527
so:0000673,0.018465,-0.033552,-0.018579,-0.010986,-0.004078,0.014320,0.030253,-0.030554,0.026595,-0.029279,...,0.046170,-0.007131,-0.010736,-0.020661,-0.002482,-0.009765,-0.006223,0.013704,0.028415,0.003572
ncbigene:55237,-0.027238,0.004067,-0.036024,-0.001426,0.017603,0.004726,0.038429,0.033189,0.005087,0.004143,...,0.032224,0.013217,-0.023572,-0.011071,0.039881,-0.027857,-0.018274,0.000782,0.039644,0.013521
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
pr:p56678,-0.014111,-0.004581,-0.018009,0.016947,-0.010388,-0.035234,-0.002214,-0.000278,-0.029503,-0.024742,...,0.011216,0.028972,0.035449,0.041572,-0.047711,0.025424,0.017413,0.023862,0.006960,0.026952
pr:p69747,-0.033205,-0.015036,-0.021176,0.024073,-0.025183,0.015103,0.021675,-0.002637,0.008651,-0.006513,...,-0.010722,-0.057686,0.030630,0.016603,0.026792,-0.022466,0.023378,0.002999,0.009855,0.011075
pr:p94298,-0.013353,-0.000970,-0.044898,-0.006525,0.010752,-0.035764,0.018990,0.012127,-0.022799,-0.009578,...,0.014862,-0.018567,0.008209,-0.001844,-0.000713,0.010288,0.028599,-0.038287,0.013122,0.005820
